In [2]:
import sys
import os
from pathlib import Path
# Thêm thư mục cha (rag-service) vào danh sách tìm kiếm của Python
notebook_dir = Path(os.getcwd())
rag_service_dir = str(notebook_dir.parent.resolve())
if rag_service_dir not in sys.path:
    sys.path.append(rag_service_dir)

import time
import re
import requests
import json
import chromadb
from typing import List, Dict, Any

from configs.setting import settings
from configs.GetConfig import config
from src.a_ingestion.a1_loader import SupabaseDataLoader
from src.b_indexing.b0_vector_db import ChromaVectorDatabase
from src.LLMService import LLMService

In [2]:
SuperbaseDataLoader = SupabaseDataLoader()
products = SuperbaseDataLoader.load_products()
policies = SuperbaseDataLoader.load_policies()

[Loader] Đang tải dữ liệu sản phẩm từ Supabase...
  -> Đã tải batch 0 - 999 (1000 sản phẩm)
  -> Đã tải batch 1000 - 1324 (325 sản phẩm)
[Loader] Đã tải thành công tổng cộng 1325 sản phẩm.
[Loader] Đang tải dữ liệu chính sách từ Supabase...
[Loader] Đã tải thành công 3 chính sách từ database.


In [3]:
# Khởi tạo kết nối ChromaDB
client = ChromaVectorDatabase()

# Tách biệt làm 2 Collection chuyên biệt
product_col = client.get_or_create_collection(name="products_collection")
policy_col = client.get_or_create_collection(name="policies_collection")

print(f"Tổng vector sản phẩm: {product_col.count()}")
print(f"Tổng vector chính sách: {policy_col.count()}")

Tổng vector sản phẩm: 1325
Tổng vector chính sách: 3


In [4]:
# 1. Import trực tiếp các hàm Tool từ d_tools
from src.d_tools import product_search, policy_search

# ==========================================
# 🔍 TEST 1: Tìm kiếm sản phẩm (Product Search)
# ==========================================
print("=== 💻 CHẠY THỬ PRODUCT SEARCH ===")
# Test với per-item filter (brand, category, max_price) và need_price_info
product_result = product_search(
    queries=[
        {"keyword": "mỏng nhẹ pin trâu", "brand": "HP", "category": "laptop", "max_price": 20000000, "need_price_info": True},
        {"keyword": "Samsung S25", "category": "phone", "include_details": True}
    ],
    limit=3
)
print(product_result)

print("\n" + "="*50 + "\n")

# ==========================================
# 📄 TEST 2: Tìm kiếm chính sách (Policy Search)
# ==========================================
print("=== 📋 CHẠY THỬ POLICY SEARCH ===")
policy_result = policy_search(key_word="đổi trả", limit=2)
print(policy_result)

=== 💻 CHẠY THỬ PRODUCT SEARCH ===
["mỏng nhẹ pin trâu"]: No matching products found.

---

["Samsung S25"]: No matching products found.


=== 📋 CHẠY THỬ POLICY SEARCH ===
Policy Document 1 [Section: General Policy] (Type: FAQ, Score: 0.1820):
Tài liệu chính sách: Chính sách đổi trả 1-đổi-1 sản phẩm công nghệ
Nội dung chi tiết:
Chính sách đổi trả sản phẩm tại Website TMĐT Điện tử:
1. Thời gian áp dụng: Trong vòng 30 ngày kể từ ngày nhận hàng thành công.
2. Điều kiện áp dụng đổi trả:
- Sản phẩm bị lỗi kỹ thuật (lỗi phần cứng) từ phía nhà sản xuất.
- Sản phẩm còn nguyên hộp, đầy đủ phụ kiện đi kèm, hóa đơn mua hàng và phiếu bảo hành.
- Sản phẩm không bị trầy xước, nứt vỡ, ẩm ướt hoặc có dấu hiệu tự ý can thiệp phần cứng.
3. Hình thức đổi trả:
- Đổi mới 1-đổi-1 sản phẩm cùng model hoặc nâng cấp lên model cao hơn (bù chênh lệch).
- Trường hợp hết hàng đổi mới, hỗ trợ hoàn tiền 100% giá trị sản phẩm trên hóa đơn mua hàng.

Policy Document 2 [Section: General Policy] (Type: FAQ, Score: 0.0132

=== 💵 CHẠY THỬ GET PRODUCT INFO ===
- **Laptop HP Victus 15-FA2451TX D17WPPA** (ID: 126343, SKU: laptop-hp-victus-15-fa2451tx-d17wppa)
  Brand: HP | Category: laptop
  In Stock (344 available) | Price: 31,190,000 VND
- **Laptop HP Victus 15-FB3116AX BX8U4PA** (ID: 110183, SKU: laptop-hp-victus-15-fb3116ax-bx8u4pa)
  Brand: HP | Category: laptop
  In Stock (70 available) | Price: 29,490,000 VND
- **Laptop HP Victus 15-FA2731TX B85LNPA** (ID: 110165, SKU: laptop-hp-victus-15-fa2731tx-b85lnpa)
  Brand: HP | Category: laptop
  In Stock (37 available) | Price: 29,990,000 VND
- **iPhone 16 Pro 512GB | Chính hãng VN/A** (ID: 90119, SKU: iphone-16-pro-512gb)
  Brand: Apple | Category: phone
  Out of Stock | Price: 37,990,000 VND
- **iPhone 16 Pro 256GB | Chính hãng VN/A** (ID: 90118, SKU: iphone-16-pro-256gb)
  Brand: Apple | Category: phone
  Out of Stock | Price: 31,990,000 VND
- **iPhone 16 Pro Max 1TB | Chính hãng VN/A** (ID: 90117, SKU: iphone-16-pro-max-1tb)
  Brand: Apple | Category: ph

In [3]:
# ==========================================================
# 📦 TEST 3: Tra cứu đơn hàng cá nhân (Order Lookup)
# ==========================================================
from src.d_tools import order_lookup
from app.core.security import supabase_admin_client

# 1. Tự động lấy 1 bản ghi đơn hàng thực tế trong database để làm dữ liệu test
print("Đang quét tìm đơn hàng test trong database...")
test_orders = supabase_admin_client.table("orders").select("id, user_id").limit(1).execute()

if test_orders.data:
    test_order_id = test_orders.data[0]["id"]
    test_user_id = test_orders.data[0]["user_id"]
    
    print(f"Đã tìm thấy đơn hàng test:")
    print(f"  - User ID: {test_user_id}")
    print(f"  - Order ID: {test_order_id}\n")
    
    # ----------------------------------------------------
    # CASE 2: Tra cứu danh sách đơn hàng gần đây (Không truyền order_id)
    # ----------------------------------------------------
    print("=== 📋 TEST CASE 2: DANH SÁCH ĐƠN HÀNG GẦN ĐÂY ===")
    history_result = order_lookup(current_user_id=test_user_id)
    print(history_result)
    
    print("\n" + "="*50 + "\n")
    
    # ----------------------------------------------------
    # CASE 1: Tra cứu chi tiết một đơn hàng cụ thể (Có truyền order_id)
    # ----------------------------------------------------
    print("=== 🔍 TEST CASE 1: CHI TIẾT ĐƠN HÀNG CỤ THỂ ===")
    detail_result = order_lookup(current_user_id=test_user_id, order_id=test_order_id)
    print(detail_result)

else:
    print("Thông báo: Cơ sở dữ liệu hiện tại chưa có đơn hàng nào để chạy thử test.")


Đang quét tìm đơn hàng test trong database...
Đã tìm thấy đơn hàng test:
  - User ID: ff641f26-3ada-47d0-9bf2-1f4b71654064
  - Order ID: 80a172ba-e9e0-47e5-a802-efba24dd14dd

=== 📋 TEST CASE 2: DANH SÁCH ĐƠN HÀNG GẦN ĐÂY ===
=== YOUR RECENT ORDERS ===

1. Order ID: 80a172ba-e9e0-47e5-a802-efba24dd14dd
   - Date: 2026-07-21T10:28:02.093553+00:00
   - Status: PENDING
   - Total: 15,990,000 VNĐ
   - Products: OPPO Reno16 F 5G 8GB 256GB


=== 🔍 TEST CASE 1: CHI TIẾT ĐƠN HÀNG CỤ THỂ ===
=== ORDER DETAILS ===
- Order ID: 80a172ba-e9e0-47e5-a802-efba24dd14dd
- Order Date: 2026-07-21T10:28:02.093553+00:00
- Status: PENDING
- Shipping Address: B22DCKH065_Vũ Gia Khải - SĐT: 0987998685 - ĐC: ;lkjhgfd, Phường Phúc Xá, Quận Ba Đình, Thành phố Hà Nội (Ghi chú: kjhgfd)
- Total Amount: 15,990,000 VNĐ
- Purchased Items:
  - OPPO Reno16 F 5G 8GB 256GB (Quantity: 1 | Price: 15,990,000 VNĐ)
